# AI Programming — Lecture 24
## VQ-VAE Image Tokens Playground

마지막 실습에서는 **VQ-VAE의 discrete latent representation**을 직접 확인합니다.

이번 Notebook의 핵심 질문은 하나입니다.

> **이미지를 정말 token으로 바꿀 수 있을까?**

Fashion-MNIST 이미지를 작은 VQ-VAE로 압축하면
$7 \times 7$ 크기의 **codebook index map**을 얻을 수 있습니다.

```text
Image
  ↓
Encoder
  ↓
Continuous latent vectors
  ↓
Nearest codebook entries
  ↓
7 × 7 code indices
  ↓
Image Tokens
  ↓
Decoder
  ↓
Reconstruction
```

### 학습 목표

- Continuous latent와 discrete latent의 차이를 설명할 수 있습니다.
- VQ-VAE의 **codebook**과 **nearest-neighbor quantization**을 이해합니다.
- $28\times28$ 이미지를 $7\times7$ discrete token map으로 변환합니다.
- Code index map을 직접 출력하고 시각화합니다.
- Token을 일부 변경하거나 두 이미지의 token을 섞어 봅니다.
- Random token map을 decode하여 왜 **prior model**이 필요한지 확인합니다.
- VQ-VAE → VQGAN → DALL·E의 연결을 이해합니다.

### 이번 실습의 재미있는 부분

```text
Original Image
→ Image Tokens
→ Token Edit
→ Token Remix
→ Random Tokens
→ Decode!
```

> VQGAN이나 DALL·E 전체를 직접 학습하지 않습니다.  
> 마지막 강의에서는 **"image token"이라는 핵심 아이디어를 직접 가지고 놀아보는 것**에 집중합니다.

## 0. 실습 환경 설정

Fashion-MNIST와 작은 convolutional VQ-VAE를 사용하므로
Google Colab의 기본 GPU(T4 등)에서 충분히 실행할 수 있습니다.

수업 시간이 부족하면 `EPOCHS = 5`로 줄여도 됩니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

# VQ-VAE 설정
LATENT_DIM = 32
NUM_CODES = 64
COMMITMENT_COST = 0.25

BATCH_SIZE = 128
EPOCHS = 10

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

gpus = tf.config.list_physical_devices("GPU")

print("TensorFlow:", tf.__version__)
print("GPU:", gpus[0].name if gpus else "사용하지 않음 (CPU)")
print("Codebook size:", NUM_CODES)
print("Code dimension:", LATENT_DIM)

## 1. Fashion-MNIST Dataset

앞선 VAE와 DCGAN 실습에서는 MNIST 숫자를 많이 사용했습니다.

마지막 실습에서는 같은 $28\times28$ 구조를 유지하면서도
조금 더 다양한 모양을 볼 수 있도록 **Fashion-MNIST**를 사용합니다.

10개 class:

```text
0  T-shirt/top
1  Trouser
2  Pullover
3  Dress
4  Coat
5  Sandal
6  Shirt
7  Sneaker
8  Bag
9  Ankle boot
```

In [ ]:
(x_train, y_train), (x_test, y_test) = (
    keras.datasets.fashion_mnist.load_data()
)

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

print("x_train:", x_train.shape)
print("x_test :", x_test.shape)
print("Pixel range:", x_train.min(), "~", x_train.max())

In [ ]:
plt.figure(figsize=(10, 2))

for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(
        x_train[i].squeeze(),
        cmap="gray"
    )
    plt.title(
        class_names[y_train[i]],
        fontsize=8
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 2. VAE와 VQ-VAE의 Latent Space 차이

### VAE

```text
Encoder
→ μ, σ
→ continuous z
→ Decoder
```

Latent variable은 연속값입니다.

### VQ-VAE

```text
Encoder
→ continuous z_e
→ nearest codebook vector
→ discrete code index
→ quantized z_q
→ Decoder
```

각 spatial location의 latent vector는
codebook에 있는 **유한한 개수의 vector 중 하나**로 바뀝니다.

이번 실습에서는:

```text
Codebook size = 64
Code dimension = 32
Latent map = 7 × 7
```

따라서 이미지 하나는 최종적으로 **49개의 integer token**으로 표현됩니다.

## 3. Encoder

$28\times28$ image를 두 번 downsampling하여
$7\times7$ latent feature map을 만듭니다.

```text
28 × 28 × 1
↓ Conv, stride 2
14 × 14 × 32
↓ Conv, stride 2
7 × 7 × 64
↓ 1×1 Conv
7 × 7 × 32
```

각 $7\times7$ 위치에는 32-dimensional continuous latent vector가 있습니다.

In [ ]:
def build_encoder():
    inputs = keras.Input(
        shape=(28, 28, 1)
    )

    x = layers.Conv2D(
        32,
        kernel_size=4,
        strides=2,
        padding="same",
        activation="relu"
    )(inputs)

    x = layers.Conv2D(
        64,
        kernel_size=4,
        strides=2,
        padding="same",
        activation="relu"
    )(x)

    outputs = layers.Conv2D(
        LATENT_DIM,
        kernel_size=1,
        padding="same"
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name="encoder"
    )


encoder = build_encoder()
encoder.summary()

## 4. Vector Quantizer와 Codebook

VQ-VAE의 핵심입니다.

Encoder output의 각 latent vector $z_e$와
codebook의 모든 vector $e_k$ 사이의 거리를 계산합니다.

$$
k^*
=
\arg\min_k
\|z_e-e_k\|_2^2
$$

가장 가까운 code를 선택합니다.

```text
continuous latent vector
        ↓ nearest neighbor
Code #17
        ↓
codebook[17]
```

선택된 `17` 같은 정수가 바로 **image token**입니다.

### Straight-Through Estimator

Nearest-neighbor 선택은 미분하기 어렵기 때문에,
backward pass에서는 quantization을 건너뛰어 gradient를 encoder로 전달합니다.

In [ ]:
class VectorQuantizer(layers.Layer):
    def __init__(
        self,
        num_codes,
        code_dim,
        commitment_cost=0.25,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_codes = num_codes
        self.code_dim = code_dim
        self.commitment_cost = commitment_cost

        self.codebook = self.add_weight(
            name="codebook",
            shape=(num_codes, code_dim),
            initializer=keras.initializers.RandomUniform(
                minval=-1.0 / num_codes,
                maxval=1.0 / num_codes
            ),
            trainable=True
        )

    def get_code_indices(self, z_e):
        # (B, H, W, D) → (B*H*W, D)
        flat_z = tf.reshape(
            z_e,
            [-1, self.code_dim]
        )

        # ||z - e||^2
        distances = (
            tf.reduce_sum(
                flat_z ** 2,
                axis=1,
                keepdims=True
            )
            + tf.reduce_sum(
                self.codebook ** 2,
                axis=1
            )
            - 2.0 * tf.matmul(
                flat_z,
                self.codebook,
                transpose_b=True
            )
        )

        indices = tf.argmin(
            distances,
            axis=1
        )

        input_shape = tf.shape(z_e)

        indices = tf.reshape(
            indices,
            [
                input_shape[0],
                input_shape[1],
                input_shape[2],
            ]
        )

        return indices

    def indices_to_embeddings(self, indices):
        return tf.gather(
            self.codebook,
            indices
        )

    def call(self, z_e):
        indices = self.get_code_indices(
            z_e
        )

        z_q = self.indices_to_embeddings(
            indices
        )

        # Codebook loss:
        # codebook은 encoder output을 따라감
        codebook_loss = tf.reduce_mean(
            (
                tf.stop_gradient(z_e)
                - z_q
            ) ** 2
        )

        # Commitment loss:
        # encoder output이 선택한 code 근처에 머물도록 함
        commitment_loss = tf.reduce_mean(
            (
                z_e
                - tf.stop_gradient(z_q)
            ) ** 2
        )

        self.add_loss(
            codebook_loss
            + self.commitment_cost
            * commitment_loss
        )

        # Straight-through estimator
        z_q_st = (
            z_e
            + tf.stop_gradient(
                z_q - z_e
            )
        )

        return z_q_st


quantizer = VectorQuantizer(
    NUM_CODES,
    LATENT_DIM,
    commitment_cost=COMMITMENT_COST,
    name="vector_quantizer"
)

## 5. Decoder

Quantized latent map을 다시 $28\times28$ image로 복원합니다.

```text
7 × 7 × 32
↓
Conv2DTranspose
↓
14 × 14 × 64
↓
Conv2DTranspose
↓
28 × 28 × 32
↓
Conv
↓
28 × 28 × 1
```

In [ ]:
def build_decoder():
    inputs = keras.Input(
        shape=(7, 7, LATENT_DIM)
    )

    x = layers.Conv2DTranspose(
        64,
        kernel_size=4,
        strides=2,
        padding="same",
        activation="relu"
    )(inputs)

    x = layers.Conv2DTranspose(
        32,
        kernel_size=4,
        strides=2,
        padding="same",
        activation="relu"
    )(x)

    outputs = layers.Conv2D(
        1,
        kernel_size=3,
        padding="same",
        activation="sigmoid"
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name="decoder"
    )


decoder = build_decoder()
decoder.summary()

## 6. VQ-VAE 연결하기

전체 구조:

```text
Image
↓
Encoder
↓
z_e
↓
Vector Quantizer
↓
z_q
↓
Decoder
↓
Reconstruction
```

Training loss는

```text
Reconstruction Loss
+ Codebook Loss
+ Commitment Loss
```

로 구성됩니다.

이번 실습에서는 reconstruction loss로 MSE를 사용합니다.

In [ ]:
vqvae_inputs = keras.Input(
    shape=(28, 28, 1)
)

z_e = encoder(
    vqvae_inputs
)

z_q = quantizer(
    z_e
)

reconstructed = decoder(
    z_q
)

vqvae = keras.Model(
    vqvae_inputs,
    reconstructed,
    name="vqvae"
)

vqvae.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=2e-3
    ),
    loss="mse"
)

vqvae.summary()

## 7. VQ-VAE 학습

Label은 사용하지 않습니다.

```text
Input  = Fashion-MNIST image
Target = 같은 image
```

즉, reconstruction을 학습하면서 동시에 discrete codebook을 학습합니다.

In [ ]:
history = vqvae.fit(
    x_train,
    x_train,
    validation_data=(
        x_test,
        x_test
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

## 8. Learning Curve

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history.history["loss"],
    label="Train Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.title("VQ-VAE Training")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Reconstruction

먼저 VQ-VAE가 입력 이미지를 얼마나 잘 복원하는지 확인합니다.

```text
Original
→ Encoder
→ Discrete Codes
→ Decoder
→ Reconstruction
```

In [ ]:
n = 10

recon = vqvae.predict(
    x_test[:n],
    verbose=0
)

plt.figure(figsize=(12, 3))

for i in range(n):
    plt.subplot(2, n, i + 1)
    plt.imshow(
        x_test[i].squeeze(),
        cmap="gray"
    )
    plt.axis("off")

    plt.subplot(2, n, n + i + 1)
    plt.imshow(
        recon[i].squeeze(),
        cmap="gray"
    )
    plt.axis("off")

plt.suptitle(
    "Top: Original | Bottom: Reconstruction"
)

plt.tight_layout()
plt.show()

## 10. 이미지를 정말 Token으로 바꾸기

이제 가장 중요한 부분입니다.

Test image 하나를 encoder에 넣고
각 latent 위치에서 선택된 **codebook index**를 확인합니다.

출력은 $7\times7$ integer matrix입니다.

이 49개의 정수가 바로 **image tokens**입니다.

In [ ]:
sample_id = 0

sample_image = x_test[
    sample_id:sample_id + 1
]

z_e_sample = encoder.predict(
    sample_image,
    verbose=0
)

token_map = quantizer.get_code_indices(
    z_e_sample
).numpy()[0]

print(
    "Class:",
    class_names[y_test[sample_id]]
)

print(
    "Token map shape:",
    token_map.shape
)

print("\n7 × 7 Image Token Map:")
print(token_map)

In [ ]:
plt.figure(figsize=(8, 3))

plt.subplot(1, 3, 1)
plt.imshow(
    sample_image[0].squeeze(),
    cmap="gray"
)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(
    token_map,
    cmap="tab20"
)
plt.title("7×7 Image Tokens")
plt.colorbar(
    fraction=0.046,
    pad=0.04
)

plt.subplot(1, 3, 3)
plt.imshow(
    recon[sample_id].squeeze(),
    cmap="gray"
)
plt.title("Reconstruction")
plt.axis("off")

plt.tight_layout()
plt.show()

## 11. 여러 이미지의 Token Map 보기

서로 다른 옷이나 신발이 어떤 token pattern으로 바뀌는지 비교합니다.

Token number 자체에 사람이 읽을 수 있는 의미가 있는 것은 아닙니다.
중요한 것은 **같은 codebook을 모든 이미지가 공유한다**는 점입니다.

In [ ]:
sample_ids = [0, 1, 2, 3, 4]

images = x_test[sample_ids]

z_e_multi = encoder.predict(
    images,
    verbose=0
)

token_maps = quantizer.get_code_indices(
    z_e_multi
).numpy()

plt.figure(figsize=(10, 4))

for i, idx in enumerate(sample_ids):
    plt.subplot(2, 5, i + 1)
    plt.imshow(
        images[i].squeeze(),
        cmap="gray"
    )
    plt.title(
        class_names[y_test[idx]],
        fontsize=8
    )
    plt.axis("off")

    plt.subplot(2, 5, 5 + i + 1)
    plt.imshow(
        token_maps[i],
        cmap="tab20"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 12. Codebook Usage

64개의 code가 모두 비슷하게 사용될까요?

Training image 일부를 encoding한 뒤
각 code index가 얼마나 자주 선택되는지 histogram을 그립니다.

이것은 codebook이 실제로 어떻게 사용되고 있는지 보는 간단한 방법입니다.

In [ ]:
usage_images = x_train[:5000]

z_e_usage = encoder.predict(
    usage_images,
    batch_size=256,
    verbose=0
)

usage_indices = quantizer.get_code_indices(
    z_e_usage
).numpy()

counts = np.bincount(
    usage_indices.ravel(),
    minlength=NUM_CODES
)

used_codes = np.sum(counts > 0)

print(
    f"Used codes: {used_codes} / {NUM_CODES}"
)

plt.figure(figsize=(10, 4))

plt.bar(
    np.arange(NUM_CODES),
    counts
)

plt.xlabel("Code Index")
plt.ylabel("Frequency")
plt.title("Codebook Usage")
plt.grid(
    axis="y",
    alpha=0.2
)
plt.show()

# Part II. Image Tokens Playground

## 13. Token Map을 Decoder에 직접 넣기

이제 encoder를 거치지 않고
**integer token map을 직접 decoder에 넣어** 이미지를 만들겠습니다.

```text
Code indices
↓
Codebook lookup
↓
Quantized latent vectors
↓
Decoder
↓
Image
```

In [ ]:
def decode_token_maps(token_maps):
    token_maps = tf.convert_to_tensor(
        token_maps,
        dtype=tf.int32
    )

    quantized = (
        quantizer
        .indices_to_embeddings(
            token_maps
        )
    )

    images = decoder.predict(
        quantized,
        verbose=0
    )

    return images

## 14. Token Edit — 일부 Token 바꿔보기

첫 번째 image의 token map 일부를
다른 image의 token으로 교체해 봅니다.

Pixel을 직접 바꾸는 것이 아니라
**latent token을 수정한 뒤 decoder가 다시 이미지를 만듭니다.**

In [ ]:
# 서로 다른 class의 example 찾기
idx_a = np.where(y_test == 7)[0][0]   # Sneaker
idx_b = np.where(y_test == 8)[0][0]   # Bag

images_ab = x_test[
    [idx_a, idx_b]
]

z_e_ab = encoder.predict(
    images_ab,
    verbose=0
)

tokens_ab = quantizer.get_code_indices(
    z_e_ab
).numpy()

tokens_a = tokens_ab[0]
tokens_b = tokens_ab[1]

# 중앙 3×3 영역을 B의 token으로 교체
edited_tokens = tokens_a.copy()

edited_tokens[
    2:5,
    2:5
] = tokens_b[
    2:5,
    2:5
]

edited_image = decode_token_maps(
    edited_tokens[None, ...]
)[0]

In [ ]:
plt.figure(figsize=(12, 3))

plt.subplot(1, 4, 1)
plt.imshow(
    images_ab[0].squeeze(),
    cmap="gray"
)
plt.title("A: Sneaker")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(
    tokens_a,
    cmap="tab20"
)
plt.title("A Tokens")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(
    edited_tokens,
    cmap="tab20"
)
plt.title("Edited Tokens")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(
    edited_image.squeeze(),
    cmap="gray"
)
plt.title("Decoded Edit")
plt.axis("off")

plt.tight_layout()
plt.show()

## 15. Token Remix — 두 이미지 섞기

두 이미지의 token map 절반씩을 섞어 봅니다.

```text
Left tokens from A
+
Right tokens from B
        ↓
Mixed token map
        ↓
Decoder
        ↓
?
```

결과가 반드시 자연스러울 필요는 없습니다.
오히려 왜 **좋은 image-token prior가 필요한지** 생각해 볼 수 있습니다.

In [ ]:
remix_tokens = tokens_a.copy()

# 오른쪽 절반을 B의 token으로 교체
remix_tokens[:, 4:] = (
    tokens_b[:, 4:]
)

remix_image = decode_token_maps(
    remix_tokens[None, ...]
)[0]

plt.figure(figsize=(12, 3))

plt.subplot(1, 4, 1)
plt.imshow(
    images_ab[0].squeeze(),
    cmap="gray"
)
plt.title("A: Sneaker")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(
    images_ab[1].squeeze(),
    cmap="gray"
)
plt.title("B: Bag")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(
    remix_tokens,
    cmap="tab20"
)
plt.title("Remixed Tokens")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(
    remix_image.squeeze(),
    cmap="gray"
)
plt.title("Decoded Remix")
plt.axis("off")

plt.tight_layout()
plt.show()

## 16. Random Tokens → Decode

이제 가장 중요한 실험입니다.

Codebook index를 완전히 무작위로 선택하여
$7\times7$ token map을 만듭니다.

```text
Random token indices
→ Codebook
→ Decoder
→ Image
```

### 예상

각 token은 실제로 학습된 code이므로
local feature 자체는 어느 정도 의미가 있을 수 있습니다.

하지만 **token 간 spatial structure를 전혀 고려하지 않았기 때문에**
전체 이미지는 자연스럽지 않을 가능성이 큽니다.

바로 이 때문에 VQ-VAE에서는 PixelCNN prior,
VQGAN/DALL·E에서는 Transformer prior와 같은
**token prior model**이 필요합니다.

In [ ]:
num_random = 12

random_token_maps = np.random.randint(
    low=0,
    high=NUM_CODES,
    size=(
        num_random,
        7,
        7
    ),
    dtype=np.int32
)

random_images = decode_token_maps(
    random_token_maps
)

plt.figure(figsize=(10, 5))

for i in range(num_random):
    plt.subplot(3, 4, i + 1)
    plt.imshow(
        random_images[i].squeeze(),
        cmap="gray"
    )
    plt.axis("off")

plt.suptitle(
    "Uniform Random Image Tokens"
)

plt.tight_layout()
plt.show()

## 17. 왜 PixelCNN 또는 Transformer Prior가 필요한가?

### VQ-VAE

Stage 1:

```text
Image
→ Encoder
→ Discrete Image Tokens
→ Decoder
```

Stage 2:

```text
Image Token Sequence
→ PixelCNN Prior
→ 다음 token 예측
```

### VQGAN

VQ-VAE의 discrete token idea는 유지하지만
더 좋은 image reconstruction을 위해 perceptual/adversarial learning을 사용합니다.

Prior는 PixelCNN 대신 **Transformer**로 발전합니다.

```text
Image Tokens
→ Transformer
→ Next Image Token
```

### DALL·E

Text token을 image token 앞에 condition으로 붙입니다.

```text
Text Tokens
+
Image Tokens
      ↓
Autoregressive Transformer
      ↓
Next Image Token
      ↓
Image Decoder
```

즉, 오늘 직접 본 `7×7 integer token map`이
VQGAN과 DALL·E를 이해하는 출발점입니다.

## 18. Image Token Map을 Sequence로 바꾸기

Transformer prior에 넣으려면
2D index map을 1D token sequence로 펼칠 수 있습니다.

```text
7 × 7 map
→ 49 image tokens
```

실제 VQGAN/DALL·E에서는 더 큰 codebook과 더 긴 token sequence를 사용합니다.

In [ ]:
token_sequence = token_map.flatten()

print(
    "Token sequence length:",
    len(token_sequence)
)

print(
    "First 20 image tokens:"
)

print(
    token_sequence[:20]
)

## 19. 직접 해보기

### Image Tokens

1. 다른 Fashion-MNIST sample의 $7\times7$ token map을 출력하세요.
2. 같은 class끼리는 token pattern이 어느 정도 비슷한지 살펴보세요.
3. `Codebook Usage`에서 거의 사용되지 않는 code가 있는지 확인하세요.

### Token Editing

4. `2:5, 2:5` 대신 한 token만 바꾸어 보세요.
5. 중앙이 아니라 위/아래 영역의 token을 바꾸어 보세요.
6. Sneaker + Bag 대신 다른 두 class를 remix해 보세요.

### Random Tokens

7. Uniform random tokens의 decode 결과가 왜 자연스럽지 않은지 설명하세요.
8. 실제 training image의 token map에서 일부 token만 random하게 바꾸어 보세요.

### 생각해 보기

9. PixelCNN이나 Transformer가 **raw pixels가 아니라 image token sequence**를 학습하면 어떤 장점이 있을까요?
10. DALL·E에서 text tokens와 image tokens를 같은 Transformer sequence에서 다룰 수 있는 이유를 설명해 보세요.

# 20. 마지막 정리

이번 강의의 가장 중요한 연결은 다음과 같습니다.

### VAE

```text
Image
→ Continuous Latent
→ Decoder
```

### VQ-VAE

```text
Image
→ Discrete Codebook Indices
→ Image Tokens
→ Decoder
```

### VQGAN

```text
Better Image Tokens
+
Transformer Prior
```

### DALL·E

```text
Text Tokens
+
Image Tokens
        ↓
Autoregressive Transformer
        ↓
New Image Tokens
        ↓
Image Decoder
```

### 오늘 직접 해본 것

```text
Fashion-MNIST Image
        ↓
      VQ-VAE
        ↓
  7 × 7 Image Tokens
        ↓
┌───────────────┬──────────────┬───────────────┐
│ Token Edit    │ Token Remix  │ Random Tokens │
└───────────────┴──────────────┴───────────────┘
        ↓
      Decoder
        ↓
      Image
```

### 꼭 기억할 것

1. **VQ-VAE는 continuous latent 대신 discrete codebook을 사용합니다.**
2. **Codebook index는 image token으로 해석할 수 있습니다.**
3. **Quantization은 각 latent vector를 가장 가까운 codebook vector로 바꿉니다.**
4. **Random token 조합은 spatial structure를 모르기 때문에 좋은 이미지를 만들기 어렵습니다.**
5. **그래서 image-token sequence의 구조를 학습하는 prior model이 필요합니다.**
6. **VQ-VAE의 PixelCNN prior는 VQGAN/DALL·E에서 Transformer prior로 발전합니다.**
7. **DALL·E는 text와 image를 모두 token sequence로 다룬다는 점이 핵심입니다.**

> 이것으로 AI Programming의 마지막 실습을 마칩니다.